# TextLab: Dreaddit data exploration

Open this notebook in VS Code or Jupyter with the project's `.venv` kernel
(**Select Kernel → Python Environments → .venv**). Run `python -m scripts.prepare_data`
first so `data/processed/` exists. Uses only training data for anything that guides modelling.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from ml.data import load_split, topic_summary

train, test = load_split("train"), load_split("test")
print(f"train: {len(train)} posts, test: {len(test)} posts")
train.head()

## Posts and % stress per topic

In [ ]:
pd.concat({"train": topic_summary(train), "test": topic_summary(test)}, axis=1)

## Subreddits inside each topic (training data)

In [ ]:
train.groupby(["topic", "subreddit"])["label"].agg(posts="size", stress_rate="mean").round(3)

## Post length (words) by label (training data)

In [ ]:
words = train["text"].str.split().str.len()
print(words.groupby(train["label"]).describe().round(1))

fig, ax = plt.subplots(figsize=(7, 3.5))
for label, name, color in [(0, "no stress", "#2a78d6"), (1, "stress", "#eb6834")]:
    ax.hist(words[train["label"] == label], bins=40, alpha=0.6, label=name, color=color)
ax.set_xlabel("Words per post"); ax.set_ylabel("Posts"); ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## Annotator confidence
Dreaddit's `confidence` is the share of annotators who agreed with the label.

In [ ]:
train["confidence"].describe().round(3)

## Example posts

In [ ]:
for label in (1, 0):
    print(f"--- label {label} ---")
    for text in train.loc[train["label"] == label, "text"].sample(2, random_state=42):
        print("*", text[:300], "\n")

## Words the TF-IDF model relies on most
Needs `models/tfidf/` (run `python -m scripts.train_baselines`). Positive weights push towards *stress*.

In [ ]:
import joblib

pipe = joblib.load(ROOT / "models" / "tfidf" / "model.joblib")
vocab = pipe.named_steps["tfidf"].get_feature_names_out()
weights = pd.Series(pipe.named_steps["clf"].coef_[0], index=vocab).sort_values()
pd.DataFrame({"no stress": weights.head(15).index, "stress": weights.tail(15).index[::-1]})